# 02 — Data Cleaning & Feature Engineering
Transforms raw match data into a model-ready dataset: results, points, gameweek, title gap, high-stakes flags, Drop Index, rolling features, and recency weights.

## Imports

In [2]:
import pandas as pd
import numpy as np
import soccerdata as sd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print("\n✅ Imports ready")

pandas : 3.0.5
numpy  : 2.4.6

✅ Imports ready


## Paths & Config

In [18]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROC_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROC_DATA_DIR.mkdir(parents=True, exist_ok = True)

TITLE_TEAMS = ['Arsenal','Liverpool','Manchester City','Manchester United']
PL = "ENG-Premier League"
ARTETA_SEASONS = ['1920', '2021', '2122', '2223', '2324', '2425', '2526']

# Recency weights per season — 25-26 counts 1.5x in the ML model
SEASON_WEIGHTS = {
    '1920': 1.0, '2021': 1.0, '2122': 1.0,
    '2223': 1.1, '2324': 1.2, '2425': 1.3,
    '2526': 1.5,
}

print(f"Raw data dir       : {RAW_DATA_DIR}")
print(f"Processed data dir : {PROC_DATA_DIR}")
print(f"Season weights     : {SEASON_WEIGHTS}")

Raw data dir       : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\raw
Processed data dir : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\processed
Season weights     : {'1920': 1.0, '2021': 1.0, '2122': 1.0, '2223': 1.1, '2324': 1.2, '2425': 1.3, '2526': 1.5}


## Load and Combine All 4 Teams

In [4]:
# Load all 4 title teams and stack into one DataFrame

dfs = [
    pd.read_csv(RAW_DATA_DIR / f"{team.lower().replace(' ', '_')}_raw.csv")
    for team in TITLE_TEAMS
]

df = pd.concat(dfs, ignore_index = True)

# Fix date column type
df['date'] = pd.to_datetime(df['date'])

# Sort by team then date — critical for rolling features to work correctly
df = df.sort_values(['team','date']).reset_index(drop=True)


print(f"Combined shape : {df.shape}")
print(f"Teams          : {sorted(df['team'].unique())}")
print(f"Date range     : {df['date'].min().date()} → {df['date'].max().date()}")
df.head(5)

Combined shape : (1064, 11)
Teams          : ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']
Date range     : 2019-08-09 → 2026-05-24


,league,season,game_id,date,team,opponent,venue,xG,xGA,scored,conceded
0,ENG-Premier League,1920,11650,2019-08-11 14:00:00,Arsenal,Newcastle United,away,1.133090,0.380551,1,0
1,ENG-Premier League,1920,11653,2019-08-17 12:30:00,Arsenal,Burnley,home,1.164400,1.391720,1,2
2,ENG-Premier League,1920,11670,2019-08-24 17:30:00,Arsenal,Liverpool,away,0.985542,2.788210,1,3
3,ENG-Premier League,1920,11682,2019-09-01 16:30:00,Arsenal,Tottenham,home,1.925090,1.955140,2,2
4,ENG-Premier League,1920,11691,2019-09-15 15:30:00,Arsenal,Watford,away,1.006160,2.832090,2,2


## Compute Result and Points

In [5]:
# Compute result — W/D/L from scored vs conceded

df['result'] = np.where(
    df['scored'] > df['conceded'], 'W',
    np.where(df['scored'] == df['conceded'], 'D', 'L')
)

# Compute points — 3 for win, 1 for draw, 0 for loss
df['points'] = df['result'].map({'W' : 3, 'D' : 1, 'L' : 0})

# Compute goal difference per match
df['gd'] = df['scored'] - df['conceded']

# Compute xG difference
df['xgd'] = df['xG'] - df['xGA']

print("Result distribution across all 4 teams:")
print(df['result'].value_counts())
print()
print("Points distribution:")
print(df['points'].value_counts().sort_index())
print()
print("Spot check — first 5 Arsenal rows:")
df[df['team'] == 'Arsenal'][['date', 'opponent', 'scored', 'conceded', 'result', 'points']].head(5)

Result distribution across all 4 teams:
result
L    491
W    351
D    222
Name: count, dtype: int64

Points distribution:
points
0    491
1    222
3    351
Name: count, dtype: int64

Spot check — first 5 Arsenal rows:


,date,opponent,scored,conceded,result,points
0,2019-08-11 14:00:00,Newcastle United,1,0,W,3
1,2019-08-17 12:30:00,Burnley,1,2,L,0
2,2019-08-24 17:30:00,Liverpool,1,3,L,0
3,2019-09-01 16:30:00,Tottenham,2,2,D,1
4,2019-09-15 15:30:00,Watford,2,2,D,1


## Add Gameweek

In [6]:
# Compute gameweek: rank matches chronologically within each team-season

df['gameweek'] = (
    df.groupby(['team','season'])['date']
    .rank(method = 'dense')
    .astype(int)
)

# Verify — each team in each season should have gameweeks 1 to 38
gw_check = df.groupby(['team', 'season'])['gameweek'].max().unstack(fill_value=0)
print("Max gameweek per team per season (should all be 38):")
print(gw_check)

Max gameweek per team per season (should all be 38):
season             1920  2021  2122  2223  2324  2425  2526
team                                                       
Arsenal              38    38    38    38    38    38    38
Liverpool            38    38    38    38    38    38    38
Manchester City      38    38    38    38    38    38    38
Manchester United    38    38    38    38    38    38    38


## Reconstruct Full PL Title Table

In [8]:
print("Loading full 20-team schedule from cache...")

understat = sd.Understat(leagues=PL, seasons=ARTETA_SEASONS)
full_schedule = understat.read_schedule().reset_index()

print(f"Shape: {full_schedule.shape}")
print("Done — loaded from local cache")

Loading full 20-team schedule from cache...


[08/25/26 03:02:23] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=9247519;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=9247520;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Shape: (2660, 20)
Done — loaded from local cache


In [10]:
KEEP = ['league', 'season', 'game_id', 'date', 'team', 'scored', 'conceded']

full_home = full_schedule.rename(columns={
    'home_team': 'team', 'home_goals': 'scored', 'away_goals': 'conceded'
})
full_away = full_schedule.rename(columns={
    'away_team': 'team', 'away_goals': 'scored', 'home_goals': 'conceded'
})

full_long = pd.concat([full_home[KEEP],full_away[KEEP]],ignore_index=True)
full_long['date'] = pd.to_datetime(full_long['date'])

# Compute points for each team
full_long['points'] = np.where(
    full_long['scored'] > full_long['conceded'],3,
    np.where(full_long['scored'] == full_long['conceded'],1,0)
)

# Sort chronologically within each team-season before cumsum
full_long = full_long.sort_values(['team','season','date']).reset_index(drop = True)

print(f"Full 20-team long format: {full_long.shape}")


Full 20-team long format: (5320, 8)


In [13]:
#Compute cumsum

full_long['cum_pts'] = full_long.groupby(['team','season'])['points'].cumsum()

full_long['gw'] = (
    full_long.groupby(['team','season'])['date']
    .rank(method = 'dense')
    .astype(int)
)

print("Arsenal cumulative points (first 5 GW of 2019-20):")
mask = (full_long['team'] == 'Arsenal') & (full_long['season'] == '1920')
print(full_long[mask][['date', 'gw', 'points', 'cum_pts']].head(5))

Arsenal cumulative points (first 5 GW of 2019-20):
                 date  gw  points  cum_pts
0 2019-08-11 14:00:00   1       3        3
1 2019-08-17 12:30:00   2       3        6
2 2019-08-24 17:30:00   3       0        6
3 2019-09-01 16:30:00   4       1        7
4 2019-09-15 15:30:00   5       1        8


## Compute Title Gap

In [16]:
# Each Gameweek's leader's points

full_long['leader_pts'] = (
    full_long.groupby(['season','gw'])['cum_pts']
    .transform('max')
)

#Title Gap = Leader's points - the team's points

full_long['title_gap'] = full_long['leader_pts'] - full_long['cum_pts']

#Quick check
mask = (full_long['season'] == '1920') & (full_long['gw'] == 30)
print("GW 30 2019-20 - points and title gap: ")
print(
    full_long[mask][['team','cum_pts','leader_pts','title_gap']]
    .sort_values('title_gap').head(10)
)

GW 30 2019-20 - points and title gap: 
                         team  cum_pts  leader_pts  title_gap
2765                Liverpool       83          83          0
3069          Manchester City       63          83         20
2575                Leicester       54          83         29
1397                  Chelsea       51          83         32
3335        Manchester United       46          83         37
5083  Wolverhampton Wanderers       46          83         37
4095         Sheffield United       44          83         39
1663           Crystal Palace       42          83         41
4437                Tottenham       42          83         41
29                    Arsenal       40          83         43


In [56]:
# Extracting title_gap for only our 4 temas

title_gap_df = (
    full_long[full_long['team'].isin(TITLE_TEAMS)]
    [['season','team','gw','title_gap','cum_pts','leader_pts']]
    .copy()
    .rename(columns={'gw':'gameweek'})
)

title_gap_df['season'] = title_gap_df['season'].astype('str')
print(f"Title gap table shape: {title_gap_df.shape}")
print(f"Expected: 4 teams × 7 seasons × 38 GW = {4*7*38} rows")
title_gap_df.head(5)

Title gap table shape: (1064, 6)
Expected: 4 teams × 7 seasons × 38 GW = 1064 rows


,season,team,gameweek,title_gap,cum_pts,leader_pts
0,1920,Arsenal,1,0,3,3
1,1920,Arsenal,2,0,6,6
2,1920,Arsenal,3,3,6,9
3,1920,Arsenal,4,5,7,12
4,1920,Arsenal,5,7,8,15


## Merge Title Gap into Main DataFrame

In [58]:
# Merging title gap into our main df


df = pd.merge(
    df,
    title_gap_df,
    on=['season', 'team', 'gameweek'],
    how='left'
)

print(f"Shape after merge: {df.shape}")
print(f"title_gap nulls  : {df['title_gap'].isna().sum()}")

# Spot check — Arsenal 2021-22, final gameweek
mask = (df['team'] == 'Arsenal') & (df['season'] == '2122') & (df['gameweek'] >= 36)
df[mask][['date', 'gameweek', 'opponent', 'result', 'cum_pts', 'title_gap']].sort_values('gameweek')

Shape after merge: (1064, 25)
title_gap nulls  : 0


,date,gameweek,opponent,result,cum_pts,title_gap
111,2022-05-12 18:45:00,36,Tottenham,L,66,23
112,2022-05-16 19:00:00,37,Newcastle United,L,66,24
113,2022-05-22 15:00:00,38,Everton,L,69,24


## Define High-Stakes Matches

## Build the Drop Index

## Rolling Features
**Critical:** always `.shift(1)` before `.rolling()` — no exceptions.

## Recency Weights

## Sanity Checks

## Save to Processed